# Diabetes Health Indicators

First experiment using the shared workflow in `ml_workflow.py`.
The target is the binary indicator `Diabetes_binary`.

We use `diabetes_binary_health_indicators_BRFSS2015.csv`,
which retains the imbalanced class distribution.


In [1]:
import pandas as pd

from ml_workflow import (
    evaluate_classifier,
    load_diabetes,
    split_dataset,
    train_logistic_baseline,
)


## Dataset

We check the number of samples, the number of features, missing values,
and the class distribution.


In [2]:
X, y = load_diabetes()

print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Missing feature values: {X.isna().sum().sum()}")


Samples: 253680
Features: 21
Missing feature values: 0


In [3]:
class_distribution = pd.DataFrame({
    "count": y.value_counts().sort_index(),
    "proportion": y.value_counts(normalize=True).sort_index(),
})
class_distribution.index.name = "Diabetes_binary"
class_distribution.round(3)


,count,proportion
Diabetes_binary,,
0,218334,0.861
1,35346,0.139


## Train/test split

We use 80% of the data for training and 20% for testing.
The split is stratified to preserve similar class proportions.
The random seed is fixed at 67.


In [4]:
X_train, X_test, y_train, y_test = split_dataset(
    X, y, test_size=0.2, random_state=67
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")


Training samples: 202944
Test samples: 50736


## Logistic regression

The pipeline applies median imputation and standardization before
logistic regression. The preprocessing steps are fitted on the training split.


In [5]:
model = train_logistic_baseline(X_train, y_train)


## Test results

Precision, recall, and F1-score refer to the positive class (`1`).
We also calculate the test accuracy of always predicting the majority
class identified in the training split.


In [6]:
scores = evaluate_classifier(model, X_test, y_test)

majority_class = y_train.mode().iloc[0]
majority_accuracy = y_test.eq(majority_class).mean()
print(f"Majority-class accuracy: {majority_accuracy:.3f}")

results = pd.Series(scores, name="Logistic regression").to_frame()
results.round(3)


Majority-class accuracy: 0.861


,Logistic regression
accuracy,0.862
balanced_accuracy,0.565
precision,0.522
recall,0.153
f1,0.237
roc_auc,0.818
average_precision,0.394


The positive class represents 13.9% of the dataset. Always predicting the training majority class gives 86.1% accuracy on the test split. Logistic regression reaches 86.2% accuracy, with 56.5% balanced accuracy and 15.3% recall for the positive class.

The low recall shows that this first model misses most positive examples at its default decision threshold. Further experiments will compare class weighting and other classifiers using cross-validation on the training split.


## Cross-validation

We evaluate the pipeline using five stratified folds of the training split.
The table reports the mean and standard deviation of each metric.

In [7]:
from ml_workflow import cross_validate_classifier

cv_scores = cross_validate_classifier(model, X_train, y_train)
cv_scores.agg(["mean", "std"]).T.round(3)

,mean,std
accuracy,0.864,0.001
balanced_accuracy,0.569,0.002
precision,0.536,0.008
recall,0.159,0.005
f1,0.246,0.006
roc_auc,0.823,0.002
average_precision,0.407,0.002


## Class weighting

In [8]:
balanced_model = train_logistic_baseline(
    X_train, y_train, class_weight="balanced"
)

balanced_cv_scores = cross_validate_classifier(
    balanced_model, X_train, y_train, random_state=67
)

comparison = pd.DataFrame({
    "standard": cv_scores.mean(),
    "balanced": balanced_cv_scores.mean(),
}).T

comparison.round(3)

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
standard,0.864,0.569,0.536,0.159,0.246,0.823,0.407
balanced,0.733,0.747,0.313,0.766,0.444,0.824,0.406


Class weighting increases recall from 0.159 to 0.766, while precision decreases from 0.536 to 0.313. Balanced accuracy improves from 0.569 to 0.747, and F1 increases from 0.246 to 0.444. The weighted model detects more positive cases but produces more false positives, reducing overall accuracy from 0.864 to 0.733. We retain this configuration for comparison with other classifiers.

## Comparing classifiers

In [ ]:
from ml_workflow import build_models, compare_models

SEED = 67

models = build_models(random_state=SEED)

comparison, fold_scores = compare_models(
    models,
    X_train,
    y_train,
    random_state=SEED,
)

mean_scores = comparison.xs("mean", axis=1, level=1)
mean_scores.sort_values("f1", ascending=False).round(3)